# PlantCLEF2026 Baseline with MLflow
This notebook reproduces the tiling inference baseline and tracks the experiment using the local MLflow and MinIO setup.

In [ ]:
import sys
import os

# Add the src folder to the Python path
sys.path.append(os.path.abspath('..'))

import numpy as np 
import pandas as pd
import timm 
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
import time
import torchvision.transforms as T
from torch.amp import autocast
from matplotlib import pyplot as plt
from kornia import tensor_to_image
from kornia.contrib import extract_tensor_patches, compute_padding
import csv
import mlflow

from src.config.mlflow_init import init_mlflow

In [ ]:
# Configure MLflow pointing to local MinIO & Postgres infrastructure
os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:5000"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://localhost:9000"
os.environ["AWS_ACCESS_KEY_ID"] = "minioadmin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "minioadmin"
os.environ["EXPERIMENT_NAME"] = "PlantCLEF2026"

# Initialize MLflow experiment
init_mlflow()

In [ ]:
class AverageMeter:
    def __init__(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count
        

class PatchDataset(Dataset):
    def __init__(self, patches, transform=None):
        self.patches = patches.squeeze(0)
        self.transform = transform

    def __len__(self):
        return self.patches.size(0)

    def __getitem__(self, idx):
        patch = self.patches[idx]
        
        if self.transform:
            patch = self.transform(patch)
        return patch


class TestDataset(Dataset):
    def __init__(self, image_folder, patch_size=518, stride=259, transform=None, use_pad=False):
        self.image_folder = image_folder
        self.image_paths = [os.path.join(image_folder, f) for f in os.listdir(image_folder)]
        self.transform = transform
        self.use_pad = use_pad
        self.patch_size = patch_size
        self.stride = stride
        
    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path)

        if self.transform:
            image = self.transform(image).unsqueeze(0)
        
        h, w = image.shape[-2:]
        
        if self.use_pad:
            pad = compute_padding(original_size=(h, w), window_size=self.patch_size, stride=self.stride)
            patches = extract_tensor_patches(image, self.patch_size, self.stride, padding=pad)
        else:
            patches = extract_tensor_patches(image, self.patch_size, self.stride)

        return patches, image_path

In [ ]:
df_species_ids = pd.read_csv('../data/species_ids.csv')

df_metadata = pd.read_csv('../data/PlantCLEF2024_single_plant_training_metadata.csv', sep=';', dtype={'partner': str})
class_map = df_species_ids['species_id'].to_dict() # dictionary to map the species model Id with the species Id

df_metadata.head()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_name = 'vit_base_patch14_reg4_dinov2.lvd142m'

# Initialize model
model = timm.create_model(model_name,
                          pretrained=False,
                          num_classes=len(df_species_ids),
                          checkpoint_path='../model/model_best.pth.tar') # Directed to the extracted model folder 
model = model.to(device)
model = model.eval()

data_config = timm.data.resolve_model_data_config(model)
model_input_size, model_mean, model_std = data_config['input_size'][1], data_config['mean'], data_config['std']

In [ ]:
# Hyperparameters
batch_size = 64
min_score = 0.1
top_k_tile = 2
patch_size = model_input_size
stride = int(model_input_size / 2)
use_pad = True

Submit prediction inside an MLflow run to log metrics and artifacts

In [ ]:
# Setup PyTorch Dataset and DataLoader
image_to_tensor = T.ToTensor()
dataset = TestDataset(image_folder='../data/PlantCLEF2025_test_images/PlantCLEF2025_test_images/',
                      patch_size=patch_size,
                      stride=stride,
                      use_pad=True,
                      transform=image_to_tensor)
dataloader = DataLoader(dataset, batch_size=1, num_workers=4, pin_memory=True)

image_predictions = {}
batch_time = AverageMeter()

# Enable PyTorch autologging and MLflow tracing if available
mlflow.pytorch.autolog()

# Start MLflow run
with mlflow.start_run(run_name="tiling-inference-baseline") as run:
    # Log parameters
    mlflow.log_params({
        "model_name": model_name,
        "batch_size": batch_size,
        "min_score": min_score,
        "top_k_tile": top_k_tile,
        "patch_size": patch_size,
        "stride": stride,
        "use_pad": use_pad,
        "device": device.type,
        "dataset_size": len(dataset),
    })

    # Log the full loaded model into MLflow artifact store
    mlflow.pytorch.log_model(model, "vit_dino_model")

    end = time.time()
    
    with torch.no_grad():
        for batch_idx, (patches, image_path) in enumerate(dataloader):
            image_results = {}
            quadrat_id = os.path.splitext(os.path.basename(image_path[0]))[0]
            transform_patch = T.Normalize(mean=model_mean, std=model_std)
            patch_dataset = PatchDataset(patches[0], transform=transform_patch)
            patch_loader = DataLoader(patch_dataset, batch_size=batch_size, shuffle=False)
            
            for batch_patches in patch_loader:
                batch_patches = batch_patches.to(device)
                
                with autocast(device.type):
                    outputs = model(batch_patches)
                    probabilities = torch.nn.functional.softmax(outputs, dim=1)
            
                    top_probs, top_indices = torch.topk(probabilities, top_k_tile)
                    top_probs = top_probs.cpu().numpy()
                    top_indices = top_indices.cpu().numpy()
                    
                    for top_tile_indices, top_tile_probs in zip(top_indices, top_probs):
                        for top_idx, top_prob in zip(top_tile_indices, top_tile_probs):
                            species_id = class_map[top_idx]
                            if top_prob > min_score:
                                if species_id not in image_results or image_results[species_id] < top_prob:
                                    image_results[species_id] = top_prob
            
            # Predict list
            predictions_list = list(image_results.keys())
            image_predictions[quadrat_id] = predictions_list
            
            # Batch Tracking
            num_predictions = len(predictions_list)
            max_prob = max(image_results.values()) if image_results else 0.0
            
            batch_time.update(time.time() - end)
            end = time.time()
            
            # Step metrics
            mlflow.log_metric("step_batch_time", batch_time.val, step=batch_idx)
            mlflow.log_metric("num_predictions_per_image", num_predictions, step=batch_idx)
            mlflow.log_metric("max_prob_per_image", max_prob, step=batch_idx)
    
            if batch_idx % 10 == 0:
                print(f'Predict: [{batch_idx}/{len(dataloader)}] Time {batch_time.val:.3f} ({batch_time.avg:.3f})')
                
    # Log global metrics
    mlflow.log_metric("avg_batch_time_seconds", batch_time.avg)
    mlflow.log_metric("total_images_processed", len(dataloader))
                
    # Save predictions
    submission_path = "submission.csv"
    df_run = pd.DataFrame(list(image_predictions.items()), columns=['quadrat_id', 'species_ids'])
    df_run['species_ids'] = df_run['species_ids'].apply(str)
    df_run.to_csv(submission_path, sep=',', index=False, quoting=csv.QUOTE_ALL)
    
    # Store submission.csv artifact
    mlflow.log_artifact(submission_path)
    
    print(f"Run completed. Check MLflow UI at {os.environ['MLFLOW_TRACKING_URI']}")